<a href="https://colab.research.google.com/github/Ps1012/Sales-Performance-Profitability-Analysis-Superstore/blob/main/Superstore_Sales_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup & Libraries

Loading all required libraries and defining the shared colour palette and chart layout used consistently across every chart.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Colour palette
BLUE   = '#185FA5'
GREEN  = '#1A7A4A'
RED    = '#B83232'
ORANGE = '#E07B00'
PURPLE = '#6B3FA0'
GREY   = '#6B7280'

# Shared chart layout applied to every figure
LAYOUT = dict(
    font          = dict(family='Arial, sans-serif', size=12, color='#1A1A2E'),
    paper_bgcolor = 'white',
    plot_bgcolor  = '#F8F9FA',
    title_font    = dict(size=15, color='#1A1A2E'),
    title_x       = 0.05,
    hoverlabel    = dict(bgcolor='white', font_size=12),
)
print('Libraries loaded.')

Libraries loaded.


## 2. Load the Data

Reading the CSV file and parsing date columns. This dataset requires `encoding='latin1'` due to special characters in product names.


In [ ]:
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Dates   : {df["Order Date"].min().date()} to {df["Order Date"].max().date()}')
print(f'Columns : {df.columns.tolist()}')

Shape   : 9,994 rows x 21 columns
Dates   : 2014-01-03 to 2017-12-30
Columns : ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## 3. First Look

Previewing raw data — checking structure, column types, and the first few rows before doing anything else.


In [ ]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 4. Basic Statistics

Summary statistics for all 21 columns — checking value ranges, distributions, and spotting anything unusual.


In [ ]:
df.describe(include='all')

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
count,9994.000000,9994,9994,9994,9994,9994,9994,9994,9994,9994,...,9994.000000,9994,9994,9994,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000
unique,NaN,5009,NaN,NaN,4,793,793,3,1,531,...,NaN,4,1862,3,17,1850,NaN,NaN,NaN,NaN
top,NaN,CA-2017-100111,NaN,NaN,Standard Class,WB-21850,William Brown,Consumer,United States,New York City,...,NaN,West,OFF-PA-10001970,Office Supplies,Binders,Staple envelope,NaN,NaN,NaN,NaN
freq,NaN,14,NaN,NaN,5968,37,37,5191,9994,915,...,NaN,3203,19,6026,1523,48,NaN,NaN,NaN,NaN
mean,4997.500000,NaN,2016-04-30 00:07:12.259355648,2016-05-03 23:06:58.571142912,NaN,NaN,NaN,NaN,NaN,NaN,...,55190.379428,NaN,NaN,NaN,NaN,NaN,229.858001,3.789574,0.156203,28.656896
min,1.000000,NaN,2014-01-03 00:00:00,2014-01-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,1040.000000,NaN,NaN,NaN,NaN,NaN,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,NaN,2015-05-23 00:00:00,2015-05-27 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,23223.000000,NaN,NaN,NaN,NaN,NaN,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,NaN,2016-06-26 00:00:00,2016-06-29 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,56430.500000,NaN,NaN,NaN,NaN,NaN,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,NaN,2017-05-14 00:00:00,2017-05-18 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,90008.000000,NaN,NaN,NaN,NaN,NaN,209.940000,5.000000,0.200000,29.364000
max,9994.000000,NaN,2017-12-30 00:00:00,2018-01-05 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,99301.000000,NaN,NaN,NaN,NaN,NaN,22638.480000,14.000000,0.800000,8399.976000


## 5. Data Quality Check

Checking for missing values, duplicates, and confirming the dataset is clean. Note: duplicate Order IDs are expected — each order has multiple product line items.


In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print(f'Duplicate rows    : {df.duplicated().sum()}')
print(f'Duplicate Order IDs (expected — multiple items per order):')
print(f'  Unique Orders   : {df["Order ID"].nunique():,}')
print(f'  Total rows      : {len(df):,}')
print(f'  Avg items/order : {len(df)/df["Order ID"].nunique():.1f}')


Missing values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Duplicate rows    : 0
Duplicate Order IDs (expected — multiple items per order):
  Unique Orders   : 5,009
  Total rows      : 9,994
  Avg items/order : 2.0


## 6. Feature Engineering

Creating 8 new columns before analysis begins. These derived columns power the time-series charts, margin calculations, and discount analysis downstream.

- `Year`, `Month`, `MonthName`, `YearMonth` — for time-based grouping
- `ShipDays` — days between order placed and delivered
- `ProfitMargin` — profit as a percentage of sales
- `IsLoss` — flags orders where profit is negative
- `IsDiscounted` — flags orders where any discount was applied


In [ ]:
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['MonthName']     = df['Order Date'].dt.strftime('%b')
df['YearMonth']     = df['Order Date'].dt.to_period('M').astype(str)
df['ShipDays']      = (df['Ship Date'] - df['Order Date']).dt.days
df['ProfitMargin']  = (df['Profit'] / df['Sales'] * 100).round(2)
df['IsLoss']        = df['Profit'] < 0
df['IsDiscounted']  = df['Discount'] > 0

print('New columns created:')
print('  Year, Month, MonthName, YearMonth — for time-based analysis')
print('  ShipDays      — days between order and delivery')
print('  ProfitMargin  — profit as % of sales')
print('  IsLoss        — True if order is loss-making')
print('  IsDiscounted  — True if any discount was applied')
print()
print(df[['Sales','Profit','ShipDays','ProfitMargin','IsLoss','IsDiscounted']].describe().round(2))


New columns created:
  Year, Month, MonthName, YearMonth — for time-based analysis
  ShipDays      — days between order and delivery
  ProfitMargin  — profit as % of sales
  IsLoss        — True if order is loss-making
  IsDiscounted  — True if any discount was applied

          Sales   Profit  ShipDays  ProfitMargin
count   9994.00  9994.00   9994.00       9994.00
mean     229.86    28.66      3.96         12.03
std      623.25   234.26      1.75         46.68
min        0.44 -6599.98      0.00       -275.00
25%       17.28     1.73      3.00          7.50
50%       54.49     8.67      4.00         27.00
75%      209.94    29.36      5.00         36.25
max    22638.48  8399.98      7.00         50.00


## 7. Business Overview — KPI Summary

High-level snapshot of the business: total revenue, profit, margin, orders, customers, and key risk signals (loss orders, discount rate).


In [ ]:
total_revenue   = df['Sales'].sum()
total_profit    = df['Profit'].sum()
profit_margin   = total_profit / total_revenue * 100
total_orders    = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
loss_orders     = df['IsLoss'].sum()
loss_pct        = df['IsLoss'].mean() * 100
discount_pct    = df['IsDiscounted'].mean() * 100
avg_order_value = df.groupby('Order ID')['Sales'].sum().mean()

metrics = [
    ('Total Revenue',     f'${total_revenue/1e6:.2f}M',        BLUE),
    ('Total Profit',      f'${total_profit/1e3:.1f}K',         GREEN),
    ('Profit Margin',     f'{profit_margin:.1f}%',              ORANGE),
    ('Unique Orders',     f'{total_orders:,}',                  PURPLE),
    ('Customers',         f'{total_customers:,}',               BLUE),
    ('Avg Order Value',   f'${avg_order_value:.0f}',            GREEN),
    ('Loss Orders',       f'{loss_orders:,} ({loss_pct:.1f}%)', RED),
    ('Discounted Orders', f'{discount_pct:.1f}%',               ORANGE),
]

fig = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'indicator'}] * 4] * 2,
    vertical_spacing=0.1,
)

for i, (label, value, color) in enumerate(metrics):
    row = i // 4 + 1
    col = i %  4 + 1
    fig.add_trace(
        go.Indicator(
            mode   = 'number',
            value  = 0,
            number = dict(valueformat='', suffix='', font=dict(color='white', size=1)),
            title  = dict(
                text=(
                    f'<span style="font-size:24px; font-weight:bold; color:{color}">'
                    f'{value}</span><br>'
                    f'<span style="font-size:12px; color:{GREY}">{label}</span>'
                ),
                font=dict(size=14),
            ),
            domain = dict(row=row - 1, column=col - 1),
        ),
        row=row, col=col,
    )

fig.update_layout(
    paper_bgcolor = 'white',
    plot_bgcolor  = 'white',
    height        = 240,
    margin        = dict(t=60, b=10, l=20, r=20),
    title_text    = 'Business Performance Overview — Superstore 2014–2017',
    title_font    = dict(size=15, color='#1A1A2E'),
    title_x       = 0.05,
)
fig.show()

print(f'\nKey Business Metrics:')
for label, value, _ in metrics:
    print(f'  {label:<22}: {value}')


Key Business Metrics:
  Total Revenue         : $2.30M
  Total Profit          : $286.4K
  Profit Margin         : 12.5%
  Unique Orders         : 5,009
  Customers             : 793
  Avg Order Value       : $459
  Loss Orders           : 1,871 (18.7%)
  Discounted Orders     : 52.0%


## 8. Revenue & Profit by Year

How has the business grown year on year? Comparing annual revenue and profit, and calculating year-on-year growth rate.


In [ ]:
yearly = df.groupby('Year').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique')
).reset_index()
yearly['Margin%']     = (yearly['Profit'] / yearly['Revenue'] * 100).round(1)
yearly['YoY_Revenue'] = yearly['Revenue'].pct_change() * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Annual Revenue & Profit', 'Year-on-Year Revenue Growth (%)'],
    horizontal_spacing=0.12
)

# Revenue bars
fig.add_trace(go.Bar(
    x=yearly['Year'], y=yearly['Revenue'],
    name='Revenue', marker_color=BLUE, opacity=0.85,
    text=[f'${v/1e3:.0f}K' for v in yearly['Revenue']],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>',
    width=0.35,
), row=1, col=1)

# Profit bars
fig.add_trace(go.Bar(
    x=yearly['Year'], y=yearly['Profit'],
    name='Profit', marker_color=GREEN, opacity=0.85,
    text=[f'${v/1e3:.0f}K' for v in yearly['Profit']],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>',
    width=0.35,
), row=1, col=1)

# YoY Growth bars — colour each bar individually
yoy_vals   = yearly['YoY_Revenue'].fillna(0).tolist()
yoy_colors = [GREEN if v >= 0 else RED for v in yoy_vals]
yoy_labels = ['Base year' if yearly['Year'].iloc[i] == yearly['Year'].min()
              else f'{v:+.1f}%' for i, v in enumerate(yoy_vals)]

fig.add_trace(go.Bar(
    x=yearly['Year'],
    y=yoy_vals,
    marker_color=yoy_colors,
    text=yoy_labels,
    textposition='outside',
    width=0.4,
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>YoY Growth: %{y:.1f}%<extra></extra>',
), row=1, col=2)


plot_layout.update({
    'title_text': 'Year-on-Year Performance',
    'barmode': 'group',
    'height': 420,
    'showlegend': True,
    'legend': dict(
        orientation='h',
        yanchor='bottom', y=1.08,
        xanchor='left', x=0.05,
        bgcolor='white',
        bordercolor='#E5E7EB',
        borderwidth=1,
    ),
})
layout_cell8 = {k: v for k, v in LAYOUT.items() if k != 'legend'}
fig.update_layout(
    **layout_cell8,
    title_text='Year-on-Year Performance',
    barmode='group',
    height=420,
    showlegend=False,

)
fig.update_yaxes(gridcolor='#E5E7EB')
fig.update_xaxes(gridcolor='#E5E7EB', tickmode='array',
                 tickvals=yearly['Year'].tolist())
fig.show()

print('\nYearly breakdown:')
print(yearly[['Year','Revenue','Profit','Margin%','Orders','YoY_Revenue']].round(1).to_string(index=False))



Yearly breakdown:
 Year  Revenue  Profit  Margin%  Orders  YoY_Revenue
 2014 484247.5 49544.0     10.2     969          NaN
 2015 470532.5 61618.6     13.1    1038         -2.8
 2016 609205.6 81795.2     13.4    1315         29.5
 2017 733215.3 93439.3     12.7    1687         20.4


## 9. Monthly Revenue Trend

Full 4-year monthly trend to identify seasonal patterns. Q4 (Nov–Dec) is highlighted — the business consistently peaks in these months every year.


In [ ]:
monthly = df.groupby('YearMonth').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
).reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Revenue'],
    name='Revenue', mode='lines',
    line=dict(color=BLUE, width=2),
    fill='tozeroy', fillcolor='rgba(24,95,165,0.08)',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Profit'],
    name='Profit', mode='lines',
    line=dict(color=GREEN, width=2, dash='dot'),
    hovertemplate='<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>',
))

# Highlight Nov-Dec peaks each year
for year in [2014, 2015, 2016, 2017]:
    fig.add_vrect(
        x0=f'{year}-11', x1=f'{year}-12',
        fillcolor='rgba(224,123,0,0.08)',
        layer='below', line_width=0,
        annotation_text='Q4 Peak' if year == 2014 else '',
        annotation_position='top left',
    )

fig.update_layout(
    **LAYOUT,
    title_text='Monthly Revenue & Profit Trend (2014–2017)',
    xaxis_title='Month',
    yaxis_title='Amount ($)',
    height=430,
    xaxis=dict(gridcolor='#E5E7EB', tickangle=-45, nticks=20),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

print('\nInsight: Q4 (Nov-Dec) consistently shows revenue spikes each year.')
print('Seasonal demand — business should plan inventory and staffing accordingly.')


Insight: Q4 (Nov-Dec) consistently shows revenue spikes each year.
Seasonal demand — business should plan inventory and staffing accordingly.


## 10. Category Performance

Comparing the three product categories on revenue, profit, and margin. This is where one of the most important findings in the dataset appears.


In [ ]:
cat = df.groupby('Category').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Row ID','count'),
    Customers=('Customer ID','nunique')
).reset_index()
cat['Margin%']   = (cat['Profit'] / cat['Revenue'] * 100).round(1)
cat['Revenue%']  = (cat['Revenue'] / cat['Revenue'].sum() * 100).round(1)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Revenue by Category', 'Profit by Category', 'Profit Margin %'],
    horizontal_spacing=0.1
)

colors_cat = [BLUE, GREEN, ORANGE]
for col_idx, (metric, label) in enumerate([('Revenue','Revenue ($)'),
                                             ('Profit','Profit ($)'),
                                             ('Margin%','Margin (%)')], 1):
    vals = cat[metric]
    bar_colors = [RED if v < 5 and metric == 'Margin%' else colors_cat[col_idx-1]
                  for v in vals]
    fig.add_trace(go.Bar(
        x=cat['Category'], y=vals,
        marker_color=bar_colors,
        text=[f'${v/1e3:.0f}K' if metric != 'Margin%' else f'{v}%' for v in vals],
        textposition='outside',
        showlegend=False,
        width=0.45,
        hovertemplate=f'<b>%{{x}}</b><br>{label}: %{{y:,.1f}}<extra></extra>',
    ), row=1, col=col_idx)
    fig.update_yaxes(gridcolor='#E5E7EB', row=1, col=col_idx)
    fig.update_xaxes(gridcolor='#E5E7EB', row=1, col=col_idx)

fig.update_layout(**LAYOUT, title_text='Category Performance Comparison', height=420)
fig.show()

print('\nCategory breakdown:')
print(cat[['Category','Revenue','Revenue%','Profit','Margin%','Orders']].round(1).to_string(index=False))
print()
print('KEY FINDING: Furniture generates 32.3% of revenue but only 2.5% profit margin.')
print('Office Supplies and Technology run at 17% margin — nearly 7x more profitable per dollar.')


Category breakdown:
       Category  Revenue  Revenue%   Profit  Margin%  Orders
      Furniture 741999.8      32.3  18451.3      2.5    2121
Office Supplies 719047.0      31.3 122490.8     17.0    6026
     Technology 836154.0      36.4 145454.9     17.4    1847

KEY FINDING: Furniture generates 32.3% of revenue but only 2.5% profit margin.
Office Supplies and Technology run at 17% margin — nearly 7x more profitable per dollar.


## 11. Sub-Category Deep Dive

Drilling into the 17 sub-categories to find which ones are profitable and which ones are actively losing money. Loss-making sub-categories are shown in red.


In [ ]:
sub = df.groupby(['Category','Sub-Category']).agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Row ID','count')
).reset_index()
sub['Margin%'] = (sub['Profit'] / sub['Revenue'] * 100).round(1)
sub = sub.sort_values('Profit', ascending=True)

profit_colors = [RED if v < 0 else GREEN if v > 20000 else BLUE
                 for v in sub['Profit']]

fig = go.Figure(go.Bar(
    x=sub['Profit'],
    y=sub['Sub-Category'],
    orientation='h',
    marker_color=profit_colors,
    text=[f'${v/1e3:.1f}K  ({m}%)' for v, m in zip(sub['Profit'], sub['Margin%'])],
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<br>Margin: %{text}<extra></extra>',
    customdata=sub['Margin%'],
))

fig.add_vline(x=0, line_color=GREY, line_width=1.5)

# Create a temporary layout dictionary, removing 'margin' to avoid duplication
plot_layout_sub = {k: v for k, v in LAYOUT.items() if k != 'margin'}
fig.update_layout(
    **plot_layout_sub,
    title_text='Profit by Sub-Category (Red = Loss-Making)',
    xaxis_title='Total Profit ($)',
    yaxis_title='Sub-Category',
    height=540,
    margin=dict(l=130, t=70, r=160, b=50),
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

print('\nLoss-making sub-categories:')
losses = sub[sub['Profit'] < 0][['Sub-Category','Category','Revenue','Profit','Margin%']]
print(losses.to_string(index=False))
print()
print('KEY FINDING: Tables lose $17,725 on $207K revenue (-8.6% margin).')
print('Bookcases lose $3,472. Both are Furniture — the entire category is dragged down.')


Loss-making sub-categories:
Sub-Category        Category     Revenue      Profit  Margin%
      Tables       Furniture 206965.5320 -17725.4811     -8.6
   Bookcases       Furniture 114879.9963  -3472.5560     -3.0
    Supplies Office Supplies  46673.5380  -1189.0995     -2.5

KEY FINDING: Tables lose $17,725 on $207K revenue (-8.6% margin).
Bookcases lose $3,472. Both are Furniture — the entire category is dragged down.


## 12. Top & Bottom 10 Products

Individual product-level profitability — identifying the 10 best and 10 worst performing products by total profit.


In [ ]:
prod = df.groupby('Product Name').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Row ID','count')
).reset_index()
prod['Margin%'] = (prod['Profit'] / prod['Revenue'] * 100).round(1)

top10    = prod.nlargest(10, 'Profit')
bottom10 = prod.nsmallest(10, 'Profit')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Top 10 Products by Profit', 'Bottom 10 Products by Profit'],
    horizontal_spacing=0.15
)

fig.add_trace(go.Bar(
    x=top10['Profit'],
    y=[n[:35]+'...' if len(n)>35 else n for n in top10['Product Name']],
    orientation='h',
    marker_color=GREEN,
    text=[f'${v:,.0f}' for v in top10['Profit']],
    textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bottom10['Profit'],
    y=[n[:35]+'...' if len(n)>35 else n for n in bottom10['Product Name']],
    orientation='h',
    marker_color=RED,
    text=[f'${v:,.0f}' for v in bottom10['Profit']],
    textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<extra></extra>',
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title_text='Top and Bottom 10 Products by Profit',
    height=480,
    margin=dict(l=280, t=70, r=120, b=50),
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
    xaxis2=dict(gridcolor='#E5E7EB'),
    yaxis2=dict(gridcolor='#E5E7EB'),
)
fig.show()


## 13. Regional Performance

Comparing the four sales regions on revenue, profit, and margin. The Central region underperforms significantly despite having 22% revenue share.


In [ ]:
region = df.groupby('Region').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique'),
    Customers=('Customer ID','nunique')
).reset_index()
region['Margin%']  = (region['Profit'] / region['Revenue'] * 100).round(1)
region['Revenue%'] = (region['Revenue'] / region['Revenue'].sum() * 100).round(1)
region = region.sort_values('Revenue', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'bar'}, {'type':'pie'}]],
    subplot_titles=['Revenue & Profit by Region', 'Revenue Share by Region'],
    horizontal_spacing=0.12
)

fig.add_trace(go.Bar(
    x=region['Region'], y=region['Revenue'],
    name='Revenue', marker_color=BLUE, opacity=0.8,
    text=[f'${v/1e3:.0f}K' for v in region['Revenue']],
    textposition='outside', width=0.35,
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=region['Region'], y=region['Profit'],
    name='Profit', marker_color=GREEN, opacity=0.8,
    text=[f'${v/1e3:.0f}K' for v in region['Profit']],
    textposition='outside', width=0.35,
    hovertemplate='<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=region['Region'],
    values=region['Revenue'],
    hole=0.45,
    marker_colors=[BLUE, GREEN, ORANGE, PURPLE],
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Revenue: $%{value:,.0f}<extra></extra>',
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title_text='Regional Sales Performance',
    barmode='group',
    height=420,
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

print('\nRegion breakdown:')
print(region[['Region','Revenue','Revenue%','Profit','Margin%','Orders','Customers']].round(1).to_string(index=False))
print()
print('KEY FINDING: Central region has lowest margin (7.9%) despite 21.8% revenue share.')
print('West leads in both revenue (31.6%) and margin (14.9%).')


Region breakdown:
 Region  Revenue  Revenue%   Profit  Margin%  Orders  Customers
   West 725457.8      31.6 108418.4     14.9    1611        686
   East 678781.2      29.5  91522.8     13.5    1401        674
Central 501239.9      21.8  39706.4      7.9    1175        629
  South 391721.9      17.1  46749.4     11.9     822        512

KEY FINDING: Central region has lowest margin (7.9%) despite 21.8% revenue share.
West leads in both revenue (31.6%) and margin (14.9%).


## 14. State-Level Profit Map

Geographic choropleth map of profit by US state. Red states are loss-making — identifying where the business is actively burning money despite generating sales.


In [ ]:
state = df.groupby('State').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Row ID','count')
).reset_index()
state['Margin%'] = (state['Profit'] / state['Revenue'] * 100).round(1)

# Map full state names to 2-letter abbreviations
state_abbrev = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
    'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC',
    'North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA',
    'Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN',
    'Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA',
    'West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY','District of Columbia':'DC'
}

state['StateCode'] = state['State'].map(state_abbrev)

fig = px.choropleth(
    state,
    locations='StateCode',
    locationmode='USA-states',
    color='Profit',
    color_continuous_scale=['#B83232','#F5F5F5','#1A7A4A'],
    color_continuous_midpoint=0,
    scope='usa',
    hover_name='State',
    hover_data={'Revenue': ':$,.0f', 'Profit': ':$,.0f', 'Margin%': ':.1f', 'StateCode': False},
    title='Profit by State — Red = Loss-Making',
    labels={'Profit': 'Total Profit ($)'}
)
fig.update_layout(**LAYOUT, height=450)
fig.show()

print('\nTop 5 loss-making states:')
print(state.nsmallest(5,'Profit')[['State','Revenue','Profit','Margin%']].round(1).to_string(index=False))
print()
print('Top 5 most profitable states:')
print(state.nlargest(5,'Profit')[['State','Revenue','Profit','Margin%']].round(1).to_string(index=False))
print()
print('KEY FINDING: Texas loses $25,729 despite being a large market.')
print('Ohio, Pennsylvania, and Illinois also loss-making — heavy discounting likely cause.')


Top 5 loss-making states:
         State  Revenue   Profit  Margin%
         Texas 170188.0 -25729.4    -15.1
          Ohio  78258.1 -16971.4    -21.7
  Pennsylvania 116511.9 -15560.0    -13.4
      Illinois  80166.1 -12607.9    -15.7
North Carolina  55603.2  -7490.9    -13.5

Top 5 most profitable states:
     State  Revenue  Profit  Margin%
California 457687.6 76381.4     16.7
  New York 310876.3 74038.5     23.8
Washington 138641.3 33402.7     24.1
  Michigan  76269.6 24463.2     32.1
  Virginia  70636.7 18598.0     26.3

KEY FINDING: Texas loses $25,729 despite being a large market.
Ohio, Pennsylvania, and Illinois also loss-making — heavy discounting likely cause.


## 15. Category × Region Heatmap

Which category performs best in which region? This heatmap shows profit for every category-region combination simultaneously.


In [ ]:
cat_region = df.groupby(['Region','Category'])['Profit'].sum().unstack().round(0)

fig = go.Figure(go.Heatmap(
    z=cat_region.values,
    x=cat_region.columns.tolist(),
    y=cat_region.index.tolist(),
    colorscale='RdYlGn',
    text=[[f'${v:,.0f}' for v in row] for row in cat_region.values],
    texttemplate='<b>%{text}</b>',
    textfont=dict(size=12),
    colorbar=dict(title='Profit ($)'),
    hovertemplate='<b>%{y} | %{x}</b><br>Profit: $%{z:,.0f}<extra></extra>',
))

fig.update_layout(
    **LAYOUT,
    title_text='Profit Heatmap — Region x Category',
    xaxis_title='Category',
    yaxis_title='Region',
    height=360,
)
fig.show()


## 16. Discount Impact Analysis

**This is the most important section.** Analysing how discount levels affect profitability. The finding is stark — this is what you lead with in interviews and on your resume.


In [ ]:
discount_bins = pd.cut(df['Discount'],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0],
    labels=['0% (No discount)', '1-10%', '11-20%', '21-30%', '31-40%', '41-50%', '51%+']
)
df['DiscountBand'] = discount_bins

disc_analysis = df.groupby('DiscountBand', observed=True).agg(
    Orders=('Row ID','count'),
    AvgProfit=('Profit','mean'),
    TotalProfit=('Profit','sum'),
    AvgSales=('Sales','mean'),
    LossRate=('IsLoss','mean')
).reset_index()
disc_analysis['LossRate%'] = (disc_analysis['LossRate']*100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Average Profit per Order by Discount Level',
                    'Loss-Making Order Rate by Discount Level'],
    horizontal_spacing=0.12
)

bar_colors1 = [GREEN if v > 0 else RED for v in disc_analysis['AvgProfit']]
fig.add_trace(go.Bar(
    x=disc_analysis['DiscountBand'].astype(str),
    y=disc_analysis['AvgProfit'],
    marker_color=bar_colors1,
    text=[f'${v:.1f}' for v in disc_analysis['AvgProfit']],
    textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Avg Profit: $%{y:.2f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=disc_analysis['DiscountBand'].astype(str),
    y=disc_analysis['LossRate%'],
    marker_color=[RED if v > 30 else ORANGE if v > 10 else GREEN
                  for v in disc_analysis['LossRate%']],
    text=[f'{v:.1f}%' for v in disc_analysis['LossRate%']],
    textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Loss Rate: %{y:.1f}%<extra></extra>',
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title_text='The Discount Trap — How Discounts Destroy Profit',
    height=430,
)
fig.update_xaxes(tickangle=-25, gridcolor='#E5E7EB')
fig.update_yaxes(gridcolor='#E5E7EB')
fig.add_hline(y=0, line_color=GREY, line_width=1, row=1, col=1)
fig.show()

no_disc = df[df['Discount']==0]['Profit'].mean()
with_disc = df[df['Discount']>0]['Profit'].mean()
print(f'\nAvg profit — no discount   : ${no_disc:.2f}')
print(f'Avg profit — with discount : ${with_disc:.2f}')
print(f'Difference                 : ${no_disc - with_disc:.2f} per order')
print()
print('KEY FINDING: Orders with any discount average -$6.66 profit.')
print(f'52% of all {len(df):,} transactions are discounted = massive margin erosion.')
print(disc_analysis[['DiscountBand','Orders','AvgProfit','LossRate%']].to_string(index=False))


Avg profit — no discount   : $66.90
Avg profit — with discount : $-6.66
Difference                 : $73.56 per order

KEY FINDING: Orders with any discount average -$6.66 profit.
52% of all 9,994 transactions are discounted = massive margin erosion.
    DiscountBand  Orders   AvgProfit  LossRate%
0% (No discount)    4798   66.900292        0.0
           1-10%      94   96.055074        4.3
          11-20%    3709   24.738824       14.0
          21-30%     227  -45.679636       91.6
          31-40%     233 -109.219691       88.8
          41-50%      77 -298.695314      100.0
            51%+     856  -89.438144      100.0


## 17. Discount by Category

Which categories are being discounted the most, and what is the profit impact? Scatter chart showing the relationship between discount rate and average profit per category.


In [ ]:
disc_cat = df.groupby('Category').agg(
    AvgDiscount=('Discount','mean'),
    AvgProfit=('Profit','mean'),
    LossRate=('IsLoss','mean')
).reset_index()
disc_cat['AvgDiscount%'] = (disc_cat['AvgDiscount']*100).round(1)
disc_cat['LossRate%']    = (disc_cat['LossRate']*100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=disc_cat['AvgDiscount%'],
    y=disc_cat['AvgProfit'],
    mode='markers+text',
    text=disc_cat['Category'],
    textposition='top center',
    marker=dict(size=18, color=[BLUE, GREEN, ORANGE],
                line=dict(color='white', width=2)),
    hovertemplate='<b>%{text}</b><br>Avg Discount: %{x:.1f}%<br>Avg Profit: $%{y:.2f}<extra></extra>',
))

fig.add_hline(y=0, line_color=RED, line_dash='dash',
              annotation_text='Break-even', annotation_position='right')
fig.update_layout(
    **LAYOUT,
    title_text='Average Discount % vs Average Profit — by Category',
    xaxis_title='Average Discount Applied (%)',
    yaxis_title='Average Profit per Order ($)',
    height=380,
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()


## 18. Customer Segment Performance

Comparing Consumer, Corporate, and Home Office segments on revenue, profit, margin, and average order value.


In [ ]:
seg = df.groupby('Segment').agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique'),
    Customers=('Customer ID','nunique')
).reset_index()
seg['Margin%']      = (seg['Profit'] / seg['Revenue'] * 100).round(1)
seg['AvgOrderValue']= (seg['Revenue'] / seg['Orders']).round(0)
seg['Revenue%']     = (seg['Revenue'] / seg['Revenue'].sum() * 100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Revenue & Profit by Segment', 'Profit Margin % by Segment'],
    horizontal_spacing=0.12
)

fig.add_trace(go.Bar(
    x=seg['Segment'], y=seg['Revenue'],
    name='Revenue', marker_color=BLUE, opacity=0.8,
    text=[f'${v/1e3:.0f}K' for v in seg['Revenue']],
    textposition='outside', width=0.35,
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=seg['Segment'], y=seg['Profit'],
    name='Profit', marker_color=GREEN, opacity=0.8,
    text=[f'${v/1e3:.0f}K' for v in seg['Profit']],
    textposition='outside', width=0.35,
    hovertemplate='<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=seg['Segment'], y=seg['Margin%'],
    marker_color=[ORANGE, BLUE, GREEN],
    text=[f'{v}%' for v in seg['Margin%']],
    textposition='outside',
    showlegend=False, width=0.4,
    hovertemplate='<b>%{x}</b><br>Margin: %{y:.1f}%<extra></extra>',
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title_text='Customer Segment Performance',
    barmode='group', height=420,
)
fig.update_xaxes(gridcolor='#E5E7EB')
fig.update_yaxes(gridcolor='#E5E7EB')
fig.show()

print('\nSegment breakdown:')
print(seg[['Segment','Revenue','Revenue%','Profit','Margin%','Customers','AvgOrderValue']].round(1).to_string(index=False))
print()
print('KEY FINDING: Home Office has the highest profit margin (14.0%) despite being smallest segment.')
print('Consumer is largest by revenue but has lowest margin (11.5%).')


Segment breakdown:
    Segment   Revenue  Revenue%   Profit  Margin%  Customers  AvgOrderValue
   Consumer 1161401.3      50.6 134119.2     11.5        409          449.0
  Corporate  706146.4      30.7  91979.1     13.0        236          466.0
Home Office  429653.1      18.7  60298.7     14.0        148          473.0

KEY FINDING: Home Office has the highest profit margin (14.0%) despite being smallest segment.
Consumer is largest by revenue but has lowest margin (11.5%).


## 19. Top 10 Customers by Profit

Identifying the most valuable customers. Colour-coded by segment to show which segment dominates the top customer list.


In [ ]:
cust = df.groupby(['Customer ID','Customer Name','Segment']).agg(
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    Orders=('Order ID','nunique')
).reset_index()
cust['Margin%'] = (cust['Profit']/cust['Revenue']*100).round(1)
top10_cust = cust.nlargest(10, 'Profit')

seg_colors_map = {'Consumer': BLUE, 'Corporate': GREEN, 'Home Office': ORANGE}

fig = go.Figure(go.Bar(
    x=top10_cust['Profit'],
    y=top10_cust['Customer Name'],
    orientation='h',
    marker_color=[seg_colors_map.get(s, BLUE) for s in top10_cust['Segment']],
    text=[f'${v:,.0f} ({m}%)' for v, m in zip(top10_cust['Profit'], top10_cust['Margin%'])],
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<extra></extra>',
))

fig.update_layout(
    **LAYOUT,
    title_text='Top 10 Customers by Profit',
    xaxis_title='Total Profit ($)',
    height=420,
    margin=dict(l=160, t=70, r=150, b=50),
    xaxis=dict(gridcolor='#E5E7EB'),
    yaxis=dict(gridcolor='#E5E7EB'),
)
fig.show()

## 20. Shipping Mode Analysis

Order distribution by shipping mode and average delivery days. Understanding how customers prefer to receive their orders.


In [ ]:
ship = df.groupby('Ship Mode').agg(
    Orders=('Row ID','count'),
    Revenue=('Sales','sum'),
    Profit=('Profit','sum'),
    AvgShipDays=('ShipDays','mean')
).reset_index()
ship['Margin%']  = (ship['Profit']/ship['Revenue']*100).round(1)
ship['Orders%']  = (ship['Orders']/ship['Orders'].sum()*100).round(1)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'bar'}]],
    subplot_titles=['Order Share by Ship Mode', 'Avg Shipping Days by Mode'],
    horizontal_spacing=0.12
)

fig.add_trace(go.Pie(
    labels=ship['Ship Mode'],
    values=ship['Orders'],
    hole=0.45,
    marker_colors=[BLUE, GREEN, ORANGE, PURPLE],
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Orders: %{value:,}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=ship['Ship Mode'],
    y=ship['AvgShipDays'],
    marker_color=[PURPLE, BLUE, GREEN, ORANGE],
    text=[f'{v:.1f} days' for v in ship['AvgShipDays']],
    textposition='outside',
    showlegend=False, width=0.45,
    hovertemplate='<b>%{x}</b><br>Avg Days: %{y:.1f}<extra></extra>',
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title_text='Shipping Mode Analysis',
    height=400,
)
fig.update_xaxes(gridcolor='#E5E7EB', row=1, col=2)
fig.update_yaxes(gridcolor='#E5E7EB', row=1, col=2)
fig.show()

print('\nShipping breakdown:')
print(ship[['Ship Mode','Orders','Orders%','AvgShipDays','Margin%']].round(1).to_string(index=False))
print('\n59% of orders use Standard Class (5-day shipping) — slowest option is most popular.')


Shipping breakdown:
     Ship Mode  Orders  Orders%  AvgShipDays  Margin%
   First Class    1538     15.4          2.2     13.9
      Same Day     543      5.4          0.0     12.4
  Second Class    1945     19.5          3.2     12.5
Standard Class    5968     59.7          5.0     12.1

59% of orders use Standard Class (5-day shipping) — slowest option is most popular.


## 21. Loss Order Analysis

How many orders are loss-making and which categories drive the most losses? 18% of all transactions are loss-making.


In [ ]:
loss_df  = df[df['IsLoss']]
loss_cat = loss_df.groupby('Category').agg(
    LossOrders=('Row ID','count'),
    TotalLoss=('Profit','sum')
).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Loss Orders by Category', 'Total Loss Amount by Category'],
    horizontal_spacing=0.12
)

for col_idx, (metric, label) in enumerate([('LossOrders','Number of Loss Orders'),
                                             ('TotalLoss','Total Loss ($)')], 1):
    vals = loss_cat[metric]
    fig.add_trace(go.Bar(
        x=loss_cat['Category'], y=vals,
        marker_color=RED,
        text=[f'{v:,.0f}' if col_idx==1 else f'${v:,.0f}' for v in vals],
        textposition='outside',
        showlegend=False, width=0.45,
        hovertemplate=f'<b>%{{x}}</b><br>{label}: %{{y:,.0f}}<extra></extra>',
    ), row=1, col=col_idx)
    fig.update_yaxes(gridcolor='#E5E7EB', row=1, col=col_idx)
    fig.update_xaxes(gridcolor='#E5E7EB', row=1, col=col_idx)

fig.update_layout(
    **LAYOUT,
    title_text=f'Loss Order Analysis — {len(loss_df):,} Orders ({len(loss_df)/len(df)*100:.1f}%) Are Loss-Making',
    height=400,
)
fig.show()

print(f'\nTotal loss-making orders : {len(loss_df):,} ({len(loss_df)/len(df)*100:.1f}% of all orders)')
print(f'Total loss amount        : ${loss_df["Profit"].sum():,.2f}')
print(f'\nLoss orders by category:')
print(loss_cat.to_string(index=False))



Total loss-making orders : 1,871 (18.7% of all orders)
Total loss amount        : $-156,131.29

Loss orders by category:
       Category  LossOrders   TotalLoss
      Furniture         714 -60936.1090
Office Supplies         886 -56615.2585
     Technology         271 -38579.9182


## 22. Business Summary & Recommendations

Full recap of the 5 key findings written in plain business language, followed by 6 actionable recommendations for the business.


In [ ]:
print('=' * 65)
print('  SUPERSTORE SALES ANALYSIS — FINAL SUMMARY')
print('=' * 65)

total_rev  = df['Sales'].sum()
total_prof = df['Profit'].sum()

print(f"""
Dataset
  Period      : 2014 – 2017
  Orders      : {df['Order ID'].nunique():,} unique orders | {len(df):,} line items
  Customers   : {df['Customer ID'].nunique():,}
  Revenue     : ${total_rev:,.2f}
  Profit      : ${total_prof:,.2f}
  Margin      : {total_prof/total_rev*100:.1f}%

Top 5 Findings
  1. DISCOUNT TRAP    : 52% of orders are discounted — discounted
                        orders average -$6.66 profit vs $66.90
                        for full-price orders. Discounting is the
                        #1 cause of margin erosion.

  2. FURNITURE PROBLEM: Furniture = 32.3% of revenue but only
                        2.5% margin. Tables alone lose $17,725.
                        Business should review furniture pricing
                        or eliminate loss-making lines.

  3. REGIONAL GAP     : Central region runs 7.9% margin vs
                        14.9% in West. Central likely over-
                        discounting to win orders.

  4. LOSS-MAKING STATES: Texas (-$25.7K), Ohio (-$17K),
                        Pennsylvania (-$15.6K) are major markets
                        losing money — priority for review.

  5. Q4 SEASONALITY   : Nov-Dec consistently peaks every year.
                        Business should plan inventory and
                        staffing to capture Q4 demand fully.

Business Recommendations
  → Set minimum margin thresholds before applying discounts
  → Review Furniture pricing — especially Tables sub-category
  → Investigate Central region discounting practices
  → Focus on Technology and Office Supplies (17%+ margin)
  → Build retention strategy for top 10 profitable customers
  → Push Copiers (37% margin) and Paper (43% margin) harder
""")
print('=' * 65)
print('  Analysis Complete — Prabhjot Singh')
print('=' * 65)

  SUPERSTORE SALES ANALYSIS — FINAL SUMMARY

Dataset
  Period      : 2014 – 2017
  Orders      : 5,009 unique orders | 9,994 line items
  Customers   : 793
  Revenue     : $2,297,200.86
  Profit      : $286,397.02
  Margin      : 12.5%

Top 5 Findings
  1. DISCOUNT TRAP    : 52% of orders are discounted — discounted
                        orders average -$6.66 profit vs $66.90
                        for full-price orders. Discounting is the
                        #1 cause of margin erosion.

  2. FURNITURE PROBLEM: Furniture = 32.3% of revenue but only
                        2.5% margin. Tables alone lose $17,725.
                        Business should review furniture pricing
                        or eliminate loss-making lines.

  3. REGIONAL GAP     : Central region runs 7.9% margin vs
                        14.9% in West. Central likely over-
                        discounting to win orders.

  4. LOSS-MAKING STATES: Texas (-$25.7K), Ohio (-$17K),
                        P

## 23. Export for Power BI

Exporting the enriched dataset with all engineered columns as a CSV file. This file is imported directly into Power BI to build the sales dashboard.


In [ ]:
# Export enriched dataset for Power BI dashboard
export_cols = [
    'Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
    'Customer ID', 'Customer Name', 'Segment',
    'City', 'State', 'Region',
    'Category', 'Sub-Category', 'Product Name',
    'Sales', 'Quantity', 'Discount', 'Profit',
    'Year', 'Month', 'MonthName', 'YearMonth',
    'ShipDays', 'ProfitMargin', 'IsLoss', 'IsDiscounted', 'DiscountBand'
]

df[export_cols].to_csv('superstore_enriched.csv', index=False)
print('Saved: superstore_enriched.csv')
print('Import this file into Power BI for the dashboard.')
print()
print(f'Enriched dataset: {len(df):,} rows | {len(export_cols)} columns')
print('New columns added: Year, Month, YearMonth, ShipDays,')
print('                   ProfitMargin, IsLoss, IsDiscounted, DiscountBand')

Saved: superstore_enriched.csv
Import this file into Power BI for the dashboard.

Enriched dataset: 9,994 rows | 27 columns
New columns added: Year, Month, YearMonth, ShipDays,
                   ProfitMargin, IsLoss, IsDiscounted, DiscountBand
